# 【期末專案】AI 穿搭顧問
建立一個AI穿搭聊天機器人，透過對話了解使用者需求，結合即時天氣資訊生成個人化穿搭建議，並以圖像化方式呈現穿搭結果。主要功能：
1.	多輪對話，並自動判斷對話資訊的完整性。
2.	分析使用者個人偏好（如風格、心情、場合等）。
3.	整合即時天氣資訊，給予具體的穿搭建議。
4.	根據對話歷史，自動生成穿搭圖像。


### 1. 安裝必要套件

In [ ]:
# 安裝必要套件
!pip install openai gradio requests pillow --upgrade
!pip install diffusers transformers accelerate safetensors huggingface_hub --upgrade

# 匯入函式庫
import gradio as gr
import requests
import json
import os
import torch
from datetime import datetime
from google.colab import userdata
from openai import OpenAI
from diffusers import StableDiffusionPipeline, UniPCMultistepScheduler
from PIL import Image
import gc
import random
import time
import hashlib

### 2. 設定API和初始化

*   用配置字典來統一管理所有設定，這樣以後要修改參數時只需要改一個地方就好。包括要使用的AI模型、圖片尺寸、生成步驟等重要設定。
*   使用Groq提供的Llama模型進行對話。

In [ ]:
# 系統配置
CONFIG = {
    'MODEL_NAME': "runwayml/stable-diffusion-v1-5",
    'DEFAULT_CITY': "Taipei",
    'MAX_CHAT_HISTORY': 20,
    'IMAGE_GENERATION': {
        'WIDTH': 512,
        'HEIGHT': 768,
        'INFERENCE_STEPS': 35,
        'GUIDANCE_SCALE': 8.5
    },
    'API_SETTINGS': {
        'MODEL': "llama3-70b-8192",
        'BASE_URL': "https://api.groq.com/openai/v1",
        'MAX_TOKENS': 1200,
        'TIMEOUT': 30
    }
}

# API 設定
try:
    api_key = userdata.get('Groq')
    print("成功獲取API key")
except Exception as e:
    print(f"無法獲取 API key: {e}")
    raise ValueError("API key 未設定")

model = CONFIG['API_SETTINGS']['MODEL']
base_url = CONFIG['API_SETTINGS']['BASE_URL']

os.environ['OPENAI_API_KEY'] = api_key

client = OpenAI(base_url=base_url)

### 3. 載入Stable Diffusion模型
載入用來生成穿搭圖片的AI模型。會先檢查有沒有GPU可以用，然後載入模型並進行優化設定。

In [ ]:
def load_stable_diffusion_model():
    global pipe

    print("正在載入 Stable Diffusion 模型...")

    # 檢查GPU可用性
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"使用設備: {device}")

    models_to_try = [
        "runwayml/stable-diffusion-v1-5",
        "CompVis/stable-diffusion-v1-4"
    ]

    for model_name in models_to_try:
        try:
            print(f"嘗試載入模型: {model_name}")

            pipe = StableDiffusionPipeline.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                safety_checker=None,
                requires_safety_checker=False
            ).to(device)

            pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

            # 啟用記憶體優化
            if device == "cuda":
                try:
                    pipe.enable_xformers_memory_efficient_attention()
                    print("xformers記憶體優化已啟用")
                except:
                    print("xformers記憶體優化無法啟用，使用標準模式")

                try:
                    pipe.enable_model_cpu_offload()
                    print("CPU offload已啟用")
                except:
                    print("CPU offload無法啟用")

            print(f"成功載入模型: {model_name}")
            return True, device

        except Exception as e:
            print(f"模型 {model_name} 載入失敗: {e}")
            continue

    print("所有模型載入失敗，使用純文字模式")
    return False, device

# 載入模型
model_loaded, device = load_stable_diffusion_model()
if not model_loaded:
    pipe = None

### 4. 獲取天氣資訊函數
使用免費的wttr.in API來獲取指定城市的溫度和天氣狀況。

In [ ]:
def get_weather_info(city="Taipei"):
    try:
        print(f"正在獲取 {city} 的天氣資訊...")
        url = f"https://wttr.in/{city}?format=j1"
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            data = response.json()
            current = data['current_condition'][0]

            weather_info = {
                'temperature': int(current['temp_C']),
                'description': current['weatherDesc'][0]['value']
            }
            print(f"天氣資訊獲取成功: {weather_info['temperature']}°C, {weather_info['description']}")
            return weather_info
        else:
            print(f"天氣API返回狀態碼: {response.status_code}")
            return get_default_weather()

    except requests.exceptions.Timeout:
        print("天氣API請求超時")
        return get_default_weather()
    except requests.exceptions.RequestException as e:
        print(f"天氣API請求錯誤: {e}")
        return get_default_weather()
    except (KeyError, ValueError, TypeError) as e:
        print(f"天氣資料解析錯誤: {e}")
        return get_default_weather()

# 預設天氣資訊
def get_default_weather():
    default_weather = {'temperature': 25, 'description': 'Partly cloudy'}
    print(f"使用預設天氣: {default_weather}")
    return default_weather

### 5. 對話狀態管理
追蹤對話進度和分析用戶需求。會記錄聊了幾輪、收集到哪些資訊（場合、風格、顏色偏好等），並判斷什麼時候可以給穿搭建議。

In [ ]:
# 對話狀態管理氣
class ConversationStateManager:
    def __init__(self):
        self.conversation_turns = 0
        self.collected_info = {}
        print("對話管理器已初始化")

    # 分析使用者輸入，並提取資訊
    def analyze_user_input(self, message):
        message_lower = message.lower()
        info_found = {}

        # 場合識別
        occasion_keywords = {
            '工作': ['工作', '上班', '辦公', '會議', '面試', '商務'],
            '約會': ['約會', '見面', '交往', '男友', '女友', '對象'],
            '聚會': ['聚會', '派對', '生日', '慶祝', '朋友', '同學會'],
            '休閒': ['逛街', '購物', '散步', '咖啡', '電影', '放鬆', '休息'],
            '運動': ['運動', '健身', '跑步', '瑜珈', '球類', '游泳'],
            '正式': ['正式', '典禮', '婚禮', '畢業', '頒獎', '重要']
        }

        for occasion, keywords in occasion_keywords.items():
            if any(keyword in message_lower for keyword in keywords):
                info_found['occasion'] = occasion
                break

        # 風格偏好識別
        style_keywords = {
            '甜美': ['甜美', '可愛', '少女', '粉嫩', '蕾絲', '花朵'],
            '帥氣': ['帥氣', '酷', '個性', '中性', '俐落', '簡潔'],
            '優雅': ['優雅', '氣質', '知性', '溫柔', '淑女'],
            '休閒': ['休閒', '舒適', '輕鬆', '簡單', '自然'],
            '正式': ['正式', '專業', '嚴謹', '端莊']
        }

        for style, keywords in style_keywords.items():
            if any(keyword in message_lower for keyword in keywords):
                info_found['style'] = style
                break

        # 顏色偏好識別
        color_keywords = {
            '黑色': ['黑色', '黑', '全黑'],
            '白色': ['白色', '白', '純白'],
            '藍色': ['藍色', '藍', '海軍藍', '深藍'],
            '紅色': ['紅色', '紅', '酒紅'],
            '粉色': ['粉色', '粉紅', '粉'],
            '灰色': ['灰色', '灰'],
            '綠色': ['綠色', '綠'],
            '黃色': ['黃色', '黃']
        }

        found_colors = []
        for color, keywords in color_keywords.items():
            if any(keyword in message_lower for keyword in keywords):
                found_colors.append(color)

        if found_colors:
            info_found['colors'] = found_colors

        # 心情識別
        mood_keywords = {
            '緊張': ['緊張', '焦慮', '不安', '擔心'],
            '興奮': ['興奮', '開心', '期待', '雀躍'],
            '自信': ['自信', '帥氣', '展現', '突出'],
            '放鬆': ['放鬆', '舒適', '輕鬆', '隨意'],
            '浪漫': ['浪漫', '溫柔', '甜蜜']
        }

        for mood, keywords in mood_keywords.items():
            if any(keyword in message_lower for keyword in keywords):
                info_found['mood'] = mood
                break

        return info_found

    # 更新對話狀態
    def update_conversation_state(self, message, history):
        self.conversation_turns = len(history)

        # 分析當前訊息
        current_info = self.analyze_user_input(message)

        # 更新收集到的資訊
        self.collected_info.update(current_info)

        # 分析歷史對話
        for user_msg, _ in history:
            if user_msg:
                historical_info = self.analyze_user_input(user_msg)
                for key, value in historical_info.items():
                    if key not in self.collected_info:
                        self.collected_info[key] = value

        print(f"對話狀態更新 - 輪次: {self.conversation_turns}, 收集資訊: {self.collected_info}")
        return current_info

    def should_give_detailed_advice(self):

        has_basic_info = ('occasion' in self.collected_info or
                         'style' in self.collected_info or
                         'mood' in self.collected_info)

        sufficient_turns = self.conversation_turns >= 3  # 至少3輪對話
        has_occasion_and_style = (('occasion' in self.collected_info) and
                                 ('style' in self.collected_info or
                                  'mood' in self.collected_info or
                                  'colors' in self.collected_info))

        # 資訊不足時也給建議
        long_conversation = self.conversation_turns >= 5

        should_give = (has_occasion_and_style or long_conversation or
                      (sufficient_turns and len(self.collected_info) >= 2))

        print(f"詳細建議判斷 - 基本資訊:{has_basic_info}, 足夠輪次:{sufficient_turns}, "
              f"場合+風格:{has_occasion_and_style}, 長對話:{long_conversation} -> {should_give}")

        return should_give

    def get_conversation_summary(self):
        """獲取對話摘要"""
        summary = {
            'turns': self.conversation_turns,
            'info_collected': len(self.collected_info),
            'has_occasion': 'occasion' in self.collected_info,
            'has_style_preference': any(key in self.collected_info
                                     for key in ['style', 'mood', 'colors']),
            'readiness_score': self.calculate_readiness_score()
        }
        return summary

    def calculate_readiness_score(self):
        score = 0

        if 'occasion' in self.collected_info:
            score += 40
        if 'style' in self.collected_info:
            score += 25
        if 'mood' in self.collected_info:
            score += 15
        if 'colors' in self.collected_info:
            score += 10

        score += min(self.conversation_turns * 5, 20)

        return min(score, 100)

conversation_manager = ConversationStateManager()

### 6. 對話分析函數
分析用戶是否提到具體場合、心情、風格等

In [ ]:
def analyze_conversation_context(history, current_message):

    # 合併所有使用者訊息
    all_user_messages = [msg for msg, _ in history if msg] + [current_message]
    combined_text = " ".join(all_user_messages)

    system_prompt = """你是時尚顧問，分析用戶對話中是否明確提到相關資訊。請使用台灣人習慣的中文分析。

1. **場合類型**：上班、約會、聚會、運動、購物、旅遊、面試、婚禮等
2. **心情感受**：緊張、興奮、放鬆、自信、害羞、開心、沮喪等
3. **風格偏好**：簡約、甜美、帥氣、優雅、個性、復古、街頭、正式等
4. **顏色偏好**：是否提到喜歡或不喜歡的顏色
5. **身材考量**：是否提到想修飾或突出某個部位
6. **預算考量**：是否提到價格或預算限制

**嚴格規則：只分析明確提到的內容，不要推測！**

JSON格式回覆：
{
    "has_occasion": true/false,
    "occasion": "場合" (沒有就空字串),
    "has_mood": true/false,
    "mood": "心情" (沒有就空字串),
    "has_style": true/false,
    "style": "風格" (沒有就空字串),
    "has_color_preference": true/false,
    "color_preference": "顏色偏好" (沒有就空字串),
    "has_body_concern": true/false,
    "body_concern": "身材考量" (沒有就空字串),
    "has_budget_concern": true/false,
    "budget_concern": "預算考量" (沒有就空字串),
    "has_enough_info": true/false,
    "conversation_depth": "shallow/medium/deep",
    "missing_info": ["還需要了解的資訊"]
}

記住：沒有明確提到的就不要假設！請使用台灣人習慣的中文進行分析。"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"分析這段對話：{combined_text}"}
            ],
            max_tokens=400,
            timeout=CONFIG['API_SETTINGS']['TIMEOUT']
        )

        result = response.choices[0].message.content
        try:
            return json.loads(result)
        except json.JSONDecodeError:
            print("JSON解析失敗，使用預設分析")
            return get_default_analysis(history)
    except Exception as e:
        print(f"對話分析API錯誤: {e}")
        return get_default_analysis(history)

# 預設
def get_default_analysis(history):
    return {
        "has_occasion": False,
        "occasion": "",
        "has_mood": False,
        "mood": "",
        "has_style": False,
        "style": "",
        "has_color_preference": False,
        "color_preference": "",
        "has_body_concern": False,
        "body_concern": "",
        "has_budget_concern": False,
        "budget_concern": "",
        "has_enough_info": len(history) >= 2,
        "conversation_depth": "shallow",
        "missing_info": ["需要了解你的需求"]
    }


### 7. 對話回應生成函數
如果收集到足夠資訊就給詳細穿搭建議，如果還不夠就繼續聊天收集更多需求。使用Groq API呼叫Llama3來生成自然的中文回應。

In [ ]:
def generate_casual_response(user_message, context_analysis, weather):

    system_prompt = f"""你是一位親切的台灣穿搭顧問，請以台灣人熟悉的中文，跟使用者很自然的聊天，不要太正式。請使用台灣人習慣的中文。

現在天氣：{weather['temperature']}°C，{weather['description']}

對話原則：
1. 回應要簡短（1-2句話就好）
2. 用台灣人習慣的中文
3. 不要預設任何風格或場合
4. 如果需要更多資訊，自然地問1個問題就好
5. 語氣要親切但不要太興奮

根據用戶說的話自然回應，如果資訊不夠就問一個簡單的問題。"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            max_tokens=150,
            timeout=CONFIG['API_SETTINGS']['TIMEOUT']
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ 日常對話生成錯誤: {e}")
        return "好的，讓我想想適合你的穿搭～"

def generate_final_recommendation(weather, user_context, history, gender):

    # 從對話管理器獲取更完整的資訊
    global conversation_manager
    collected_info = conversation_manager.collected_info
    conversation_summary = conversation_manager.get_conversation_summary()

    conversation_content = ""
    user_statements = []

    for user_msg, bot_response in history[-5:]:  # 取最近5輪對話
        if user_msg:
            user_statements.append(user_msg)
            conversation_content += f"用戶: {user_msg}\n"
        if bot_response:
            conversation_content += f"顧問: {bot_response[:100]}...\n"

    # 整理收集到的完整資訊
    info_analysis = analyze_collected_information(collected_info, user_statements)

    temp = weather['temperature']
    season_context = get_season_context(temp)

    system_prompt = f"""你是專業的台灣時尚造型師，根據與客戶的深度對話，給出最適合的穿搭建議。請使用台灣人習慣的中文。

**天氣資訊：**
溫度：{temp}°C ({season_context['description']})
天氣：{weather['description']}
{season_context['clothing_guidance']}

**客戶資訊分析：**
- 對話輪次：{conversation_summary['turns']}輪
- 資訊完整度：{conversation_summary['readiness_score']}%
- 主要需求：{info_analysis['primary_needs']}
- 風格偏好：{info_analysis['style_preferences']}
- 個人特質：{info_analysis['personal_traits']}

**完整對話內容：**
{conversation_content}

**收集到的具體資訊：**
{format_collected_info_for_prompt(collected_info)}

請提供詳細且個人化的穿搭建議，格式如下，請使用台灣人習慣的中文：

## 🎯 為你量身打造的穿搭方案

**整體風格定調**
(根據對話內容分析客戶的個性和需求，給出整體穿搭理念)

## 👕 上半身建議
**內搭選擇：**
- 推薦單品：(具體款式和材質)
- 顏色建議：(結合客戶偏好和膚色)
- 版型建議：(考慮身型和場合)

**外搭層次：**
- 主要外套：(根據天氣和場合)
- 材質選擇：(舒適度和實用性)
- 搭配技巧：(如何穿出層次感)

## 👖 下半身建議
**主要單品：**
- 推薦選擇：(褲裝/裙裝的具體建議)
- 版型分析：(最適合的剪裁)
- 長度建議：(配合身材比例)

**材質與色彩：**
- 面料選擇：(考慮季節和舒適度)
- 顏色搭配：(與上半身的和諧搭配)

## 👠 鞋履配件
**鞋款建議：**
- 主推鞋型：(考慮場合和舒適度)
- 顏色選擇：(整體搭配和諧度)
- 實用考量：(行程安排的適合度)

**配件加分：**
- 必備配件：(包包、飾品等)
- 細節亮點：(提升整體質感的小物)
- 季節配件：(如圍巾、帽子等)

## 🎨 色彩美學
**主色調規劃：**
- 基礎色系：(安全且實用的主色)
- 點綴色彩：(增加活力的亮點色)
- 色彩心理：(符合心情和場合的色彩效果)

## 💡 造型師小撇步
**穿搭技巧：**
(3-4個實用的穿搭小技巧，讓整體look更完美)

**場合適配：**
(針對客戶的具體場合給出注意事項)

**個人化建議：**
(根據對話中提到的個人特質給出專屬建議)

(請用溫暖親切的台灣中文，就像真正的造型師朋友在給建議一樣。每個建議都要具體實用，並解釋為什麼這樣搭配。)

**場合適配：**
(針對客戶的具體場合給出注意事項)

**個人化建議：**
(根據對話中提到的個人特質給出專屬建議)

(請用溫暖親切的台灣中文，就像真正的造型師朋友在給建議一樣。每個建議都要具體實用，並解釋為什麼這樣搭配。)"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system_prompt}],
            max_tokens=1500,  # 增加token數量以支援更詳細的建議
            timeout=CONFIG['API_SETTINGS']['TIMEOUT']
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"⚠️ 詳細建議生成錯誤: {e}")
        return generate_fallback_recommendation(weather, collected_info)

# 深度分析蒐集到的資訊
def analyze_collected_information(collected_info, user_statements):

    analysis = {
        'primary_needs': '',
        'style_preferences': '',
        'personal_traits': ''
    }

    # 分析主要需求
    if 'occasion' in collected_info:
        analysis['primary_needs'] = f"{collected_info['occasion']}場合穿搭"
    else:
        analysis['primary_needs'] = "日常穿搭提升"

    # 分析風格偏好
    style_elements = []
    if 'style' in collected_info:
        style_elements.append(collected_info['style'])
    if 'mood' in collected_info:
        style_elements.append(f"{collected_info['mood']}心情")
    if 'colors' in collected_info:
        style_elements.append(f"偏好{', '.join(collected_info['colors'][:2])}色系")

    analysis['style_preferences'] = '、'.join(style_elements) if style_elements else "探索中"

    # 分析個人特質
    full_conversation = ' '.join(user_statements).lower()
    traits = []

    if any(word in full_conversation for word in ['緊張', '不安', '擔心']):
        traits.append('需要安全感')
    if any(word in full_conversation for word in ['期待', '興奮', '開心']):
        traits.append('積極正面')
    if any(word in full_conversation for word in ['舒適', '輕鬆', '自在']):
        traits.append('重視舒適')
    if any(word in full_conversation for word in ['特別', '不一樣', '突出']):
        traits.append('喜歡獨特感')

    analysis['personal_traits'] = '、'.join(traits) if traits else "個性平衡"

    return analysis

# 依據季節給予穿搭建議
def get_season_context(temperature):

    if temperature >= 28:
        return {
            'description': '炎熱夏日',
            'clothing_guidance': '建議選擇透氣輕薄面料，注意防曬和通風，可考慮亞麻、棉質等天然纖維'
        }
    elif temperature >= 22:
        return {
            'description': '舒適春秋',
            'clothing_guidance': '適合層次穿搭，可內薄外厚，方便調節，是展現搭配功力的好季節'
        }
    elif temperature >= 15:
        return {
            'description': '涼爽秋日',
            'clothing_guidance': '需要保暖但不厚重，適合針織、薄外套，可加入秋季色彩元素'
        }
    else:
        return {
            'description': '寒冷冬日',
            'clothing_guidance': '保暖為主，但也要有型，可利用層次穿搭和配件增加造型感'
        }

# 格式化資訊-> 下prompt
def format_collected_info_for_prompt(collected_info):

    formatted_info = []

    if 'occasion' in collected_info:
        formatted_info.append(f"場合需求：{collected_info['occasion']}")

    if 'style' in collected_info:
        formatted_info.append(f"風格偏好：{collected_info['style']}")

    if 'mood' in collected_info:
        formatted_info.append(f"希望心情：{collected_info['mood']}")

    if 'colors' in collected_info:
        formatted_info.append(f"顏色偏好：{', '.join(collected_info['colors'])}")

    return '\n'.join(formatted_info) if formatted_info else "客戶偏好：正在探索中"

# 預設/備用回應
def generate_fallback_recommendation(weather, collected_info):

    temp = weather['temperature']

    if temp >= 25:
        return """## 🌞 夏日清爽穿搭建議

由於系統暫時忙碌，給你一個簡單的夏日穿搭方向：

**上半身：** 選擇透氣的棉質或亞麻襯衫，淺色系為主
**下半身：** 舒適的直筒長褲或A字裙
**鞋履：** 透氣的平底鞋或低跟涼鞋
**配件：** 遮陽帽和輕便包包

點擊「生成穿搭圖」看看實際效果！"""
    else:
        return """## 🍂 舒適保暖穿搭建議

簡單的搭配建議：

**上半身：** 針織衫配薄外套，層次穿搭
**下半身：** 合身長褲，舒適為主
**鞋履：** 包覆性好的休閒鞋或短靴
**配件：** 圍巾和實用包款

點擊「生成穿搭圖」看看實際效果！"""
        user_needs += f"• 顏色偏好：{user_context['color_preference']}\n"
    if user_context.get('has_body_concern') and user_context.get('body_concern'):
        user_needs += f"• 身材考量：{user_context['body_concern']}\n"
    if user_context.get('has_budget_concern') and user_context.get('budget_concern'):
        user_needs += f"• 預算考量：{user_context['budget_concern']}\n"

    temp = weather['temperature']
    if temp >= 28:
        season = "炎熱"
    elif temp >= 22:
        season = "舒適"
    elif temp >= 15:
        season = "涼爽"
    else:
        season = "寒冷"

    system_prompt = f"""你是專業的台灣時尚造型師，請以台灣人熟悉的中文，給客戶詳細的穿搭建議。

**基本資訊：**
天氣狀況：{temp}°C，{season}天氣，{weather['description']}
客戶性別：{gender}

**客戶需求：**{user_needs}

**最近對話：**
{conversation_summary}

請提供詳細的穿搭建議，要分點列出：

**🎯 整體造型重點**
(1-2句話說明整體感覺，結合客戶的需求)

**👕 上半身搭配**
- 內搭：
- 外套：
- 顏色建議：{f"(特別考慮：{user_context.get('color_preference', '')})" if user_context.get('has_color_preference') else ""}

**👖 下半身搭配**
- 褲裝/裙裝：{f"(考慮身材：{user_context.get('body_concern', '')})" if user_context.get('has_body_concern') else ""}
- 顏色搭配：

**👠 鞋款選擇**
- 建議鞋款：
- 顏色：

**💍 配件加分**
- 必備配件：
- 可選配件：

**🎨 色彩搭配**
- 主色調：
- 亮點色：

**💡 小提醒**
(給1-2個實用的穿搭技巧，{f"特別注意預算考量：{user_context.get('budget_concern', '')}" if user_context.get('has_budget_concern') else ""}）

用台灣人習慣的中文，語氣專業但親切，要根據客戶的具體需求給建議。"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system_prompt}],
            max_tokens=CONFIG['API_SETTINGS']['MAX_TOKENS'],
            timeout=CONFIG['API_SETTINGS']['TIMEOUT']
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"詳細建議生成錯誤: {e}")
        return f"抱歉，系統出了點問題，讓我重新為你分析穿搭建議..."

### 8. 詳細穿搭建議函數

In [ ]:
def generate_final_recommendation(weather, user_context, history, gender):
    """生成最終的詳細穿搭建議"""

    # 整理對話重點
    conversation_summary = ""
    for user_msg, _ in history[-3:]:  # 取最近3輪對話
        if user_msg:
            conversation_summary += f"• {user_msg}\n"

    # 整理用戶需求分析
    user_needs = "\n從客戶的對話中，我了解到：\n"
    if user_context.get('has_occasion') and user_context.get('occasion'):
        user_needs += f"• 場合：{user_context['occasion']}\n"
    if user_context.get('has_mood') and user_context.get('mood'):
        user_needs += f"• 心情：{user_context['mood']}\n"
    if user_context.get('has_style') and user_context.get('style'):
        user_needs += f"• 風格偏好：{user_context['style']}\n"
    if user_context.get('has_color_preference') and user_context.get('color_preference'):
        user_needs += f"• 顏色偏好：{user_context['color_preference']}\n"
    if user_context.get('has_body_concern') and user_context.get('body_concern'):
        user_needs += f"• 身材考量：{user_context['body_concern']}\n"
    if user_context.get('has_budget_concern') and user_context.get('budget_concern'):
        user_needs += f"• 預算考量：{user_context['budget_concern']}\n"

    temp = weather['temperature']
    if temp >= 28:
        season = "炎熱"
    elif temp >= 22:
        season = "舒適"
    elif temp >= 15:
        season = "涼爽"
    else:
        season = "寒冷"

    # 🔧 可修改區域 3：詳細穿搭建議prompt - 支援更多維度
    system_prompt = f"""你是專業的台灣時尚造型師，要給客戶詳細的穿搭建議。

**基本資訊：**
天氣狀況：{temp}°C，{season}天氣，{weather['description']}
客戶性別：{gender}

**客戶需求：**{user_needs}

**最近對話：**
{conversation_summary}

請提供詳細的穿搭建議，要分點列出：

**🎯 整體造型重點**
(1-2句話說明整體感覺，結合客戶的需求)

**👕 上半身搭配**
- 內搭：
- 外套：
- 顏色建議：{f"(特別考慮：{user_context.get('color_preference', '')})" if user_context.get('has_color_preference') else ""}

**👖 下半身搭配**
- 褲裝/裙裝：{f"(考慮身材：{user_context.get('body_concern', '')})" if user_context.get('has_body_concern') else ""}
- 顏色搭配：

**👠 鞋款選擇**
- 建議鞋款：
- 顏色：

**💍 配件加分**
- 必備配件：
- 可選配件：

**🎨 色彩搭配**
- 主色調：
- 亮點色：

**💡 小提醒**
(給1-2個實用的穿搭技巧，{f"特別注意預算考量：{user_context.get('budget_concern', '')}" if user_context.get('has_budget_concern') else ""}）

用台灣人習慣的中文，語氣專業但親切，要根據客戶的具體需求給建議。"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system_prompt}],
            max_tokens=1200  # 增加token數量應付更詳細的建議
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"抱歉，系統出了點問題：{str(e)}"

### 9. 回應生成函數（更精確）

In [ ]:
def generate_smart_fashion_response(message, weather, history, conversation_manager):

    try:
        print(f"處理訊息: {message[:50]}...")

        # 更新對話狀態
        current_info = conversation_manager.update_conversation_state(message, history)

        context_analysis = analyze_conversation_context(history, message)

        print(f"對話輪次: {conversation_manager.conversation_turns}")
        print(f"收集資訊: {conversation_manager.collected_info}")

        conversation_phase = determine_conversation_phase(conversation_manager, context_analysis)
        print(f"對話階段: {conversation_phase}")

        if conversation_phase == "detailed_advice":
            print("📋 生成詳細穿搭建議...")
            detailed_response = generate_final_recommendation(
                weather, context_analysis, history, "中性"
            )
            detailed_response += "\n\n💡 **準備好了嗎？** 點擊「✨ 生成我的穿搭圖」來看看實際效果！"
            return detailed_response

        else:
            print(f"生成{conversation_phase}對話...")
            return generate_conversational_response(
                message, weather, history, conversation_manager,
                context_analysis, conversation_phase
            )

    except Exception as e:
        print(f"智慧回應生成錯誤: {e}")
        return f"不好意思，我需要重新整理一下思路。可以再說一次你的需求嗎？"

# 判斷當前對話階段
def determine_conversation_phase(conversation_manager, context_analysis):

    collected_info = conversation_manager.collected_info
    turns = conversation_manager.conversation_turns

    # 檢查資訊完整度
    has_occasion = 'occasion' in collected_info and collected_info['occasion']
    has_style_info = ('style' in collected_info or 'mood' in collected_info or
                     'colors' in collected_info)
    has_sufficient_context = context_analysis.get('has_enough_info', False)

    # 階段判斷
    if has_occasion and has_style_info and turns >= 3:
        return "detailed_advice"
    elif has_occasion and not has_style_info:
        return "style_exploration"
    elif has_style_info and not has_occasion:
        return "occasion_clarification"
    elif turns >= 4:  # 聊太久但資訊不足，直接給建議
        return "detailed_advice"
    elif turns == 0:
        return "greeting"
    else:
        return "basic_exploration"

# 根據對話階段給予不同的回應策略
def generate_conversational_response(message, weather, history, conversation_manager,
                                   context_analysis, conversation_phase):

    collected_info = conversation_manager.collected_info
    temp = weather['temperature']

    if conversation_phase == "greeting":
        return generate_greeting_response(message, weather)
    elif conversation_phase == "basic_exploration":
        return generate_exploration_response(message, weather, collected_info)
    elif conversation_phase == "style_exploration":
        return generate_style_questions(message, weather, collected_info)
    elif conversation_phase == "occasion_clarification":
        return generate_occasion_questions(message, weather, collected_info)
    else:
        return generate_casual_response(message, context_analysis, weather)

# 根據天氣資訊生成問候語
def generate_greeting_response(message, weather):
    temp = weather['temperature']

    if temp >= 28:
        weather_comment = f"今天{temp}度，蠻熱的耶"
    elif temp >= 22:
        weather_comment = f"今天{temp}度，天氣還不錯"
    elif temp >= 15:
        weather_comment = f"今天{temp}度，有點涼"
    else:
        weather_comment = f"今天{temp}度，要注意保暖"

    greetings = [
        f"嗨！{weather_comment}～今天有什麼特別的安排嗎？",
        f"哈囉！{weather_comment}，想要什麼樣的穿搭呢？",
        f"Hi！{weather_comment}～要去哪裡呢？還是想聊聊穿搭風格？"
    ]

    return random.choice(greetings)

def generate_exploration_response(message, weather, collected_info):

    system_prompt = f"""你是親切的台灣穿搭顧問，用戶剛開始跟你聊天。請使用台灣人習慣的中文。

用戶說：「{message}」

現在天氣：{weather['temperature']}°C

請用台灣人習慣的中文，親切自然地回應，並且要：
1. 先回應用戶說的話
2. 問一個簡單的問題來了解更多（場合或風格偏好）
3. 保持輕鬆的語氣，1-2句話就好

範例回應風格：
- "聽起來不錯耶！是要去什麼地方嗎？"
- "我懂～想要什麼樣的感覺呢？"
- "好的！這個場合需要正式一點還是輕鬆就好？"
"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system_prompt}],
            max_tokens=100,
            timeout=CONFIG['API_SETTINGS']['TIMEOUT']
        )
        return response.choices[0].message.content.strip()
    except:
        return "聽起來不錯！想要什麼樣的感覺呢？"

# 生成風格相關問題
def generate_style_questions(message, weather, collected_info):

    occasion = collected_info.get('occasion', '未知場合')

    style_questions = [
        f"了解，{occasion}的話～你平常喜歡什麼風格呢？比較偏甜美、帥氣，還是優雅路線？",
        f"好的！{occasion}啊～想要給人什麼樣的感覺？活�潑一點還是沉穩一些？",
        f"嗯嗯，{occasion}～你比較喜歡簡約風格還是想要特別一點的搭配？",
        f"我知道了！{occasion}的話，你想要舒適自在的感覺，還是想要帥氣一點？"
    ]

    # 根據場合調整問題
    if '工作' in occasion or '面試' in occasion:
        work_questions = [
            f"工作場合啊～你們公司的穿著文化是比較正式還是casual一點？",
            f"了解！工作的話，你想要專業俐落的感覺，還是想要親和一點的風格？"
        ]
        style_questions.extend(work_questions)

    return random.choice(style_questions)

# 生成場合相關問題
def generate_occasion_questions(message, weather, collected_info):

    style_info = []
    if 'style' in collected_info:
        style_info.append(f"風格：{collected_info['style']}")
    if 'mood' in collected_info:
        style_info.append(f"心情：{collected_info['mood']}")
    if 'colors' in collected_info:
        style_info.append(f"顏色偏好：{', '.join(collected_info['colors'])}")

    style_summary = "、".join(style_info) if style_info else "你的喜好"

    occasion_questions = [
        f"我記住{style_summary}了！那今天是要去哪裡呢？",
        f"嗯嗯，{style_summary}～這樣的話，是什麼樣的場合需要穿搭建議？",
        f"好的！了解{style_summary}，那是要去上班、約會，還是其他活動呢？",
        f"收到！{style_summary}～想知道今天的行程是什麼呢？"
    ]

    return random.choice(occasion_questions)

# 生成追問問題
def generate_follow_up_questions(collected_info, conversation_turns):

    follow_ups = []

    # 根據已有資訊生成相關問題
    if 'occasion' in collected_info:
        occasion = collected_info['occasion']
        if '約會' in occasion:
            follow_ups.extend([
                "第幾次約會呢？想要給對方什麼樣的印象？",
                "約會地點是比較正式還是輕鬆的地方？",
                "想要甜美一點還是帥氣一點的感覺？"
            ])
        elif '工作' in occasion:
            follow_ups.extend([
                "是重要會議還是一般上班呢？",
                "需要特別正式嗎？",
                "想要看起來更專業還是親和一些？"
            ])
        elif '聚會' in occasion:
            follow_ups.extend([
                "是什麼樣的聚會呢？",
                "大概會有多少人？",
                "是室內還是戶外活動？"
            ])

    if 'style' in collected_info:
        style = collected_info['style']
        if '甜美' in style:
            follow_ups.extend([
                "喜歡粉嫩色系嗎？",
                "蕾絲或花朵元素可以嗎？"
            ])
        elif '帥氣' in style:
            follow_ups.extend([
                "喜歡黑白色系嗎？",
                "中性風格還是要保留一些女性化元素？"
            ])

    # 根據對話輪次決定問題深度
    if conversation_turns >= 2:
        follow_ups.extend([
            "有特別想避免的顏色嗎？",
            "身材有想要特別修飾的地方嗎？",
            "預算有考量嗎？"
        ])

    return random.choice(follow_ups) if follow_ups else "還有什麼特別想要的嗎？"

### 10. 圖片生成相關函數
將聊天內容轉換成穿搭圖片。會分析對話記錄找出用戶的顏色偏好，根據天氣決定服裝類型，然後用Stable Diffusion生成圖片。



In [ ]:
# 從對話記錄中提取AI顧問的建議
def extract_ai_recommendations(history):

    ai_recommendations = {
        'suggested_items': [],
        'suggested_colors': [],
        'suggested_styles': [],
        'styling_tips': [],
        'weather_advice': []
    }

    if not history:
        return ai_recommendations

    # 分析AI顧問的回覆
    for entry in history:
        if isinstance(entry, list) and len(entry) >= 2:
            bot_response = entry[1] if entry[1] else ""
            bot_response_lower = bot_response.lower()

            # 提取AI建議的服裝
            clothing_items = {
                'blazer': ['西裝外套', '外套', 'blazer', '正式外套'],
                'shirt': ['襯衫', '上衣', 'shirt', '襯衣'],
                'sweater': ['毛衣', '針織衫', 'sweater', '毛衫'],
                'dress': ['洋裝', '連身裙', 'dress', '裙子'],
                'pants': ['長褲', '褲子', 'pants', '西裝褲'],
                'jeans': ['牛仔褲', 'jeans', '丹寧'],
                'skirt': ['裙子', 'skirt', '短裙', '長裙'],
                'jacket': ['夾克', 'jacket', '外套'],
                'cardigan': ['開襟毛衣', 'cardigan', '罩衫'],
                'blouse': ['襯衫', 'blouse', '上衣'],
                'coat': ['大衣', 'coat', '外套', '風衣']
            }

            for item_type, keywords in clothing_items.items():
                for keyword in keywords:
                    if keyword in bot_response:
                        ai_recommendations['suggested_items'].append(item_type)
                        break

            # 提取AI建議的顏色
            ai_color_suggestions = {
                'black': ['黑色很適合', '建議選擇黑色', '黑色會很棒', '黑色系'],
                'white': ['白色乾淨', '選擇白色', '白色襯衫', '白色系'],
                'blue': ['藍色很棒', '深藍色', '海軍藍', '藍色系'],
                'gray': ['灰色優雅', '選擇灰色', '灰色系'],
                'beige': ['米色溫和', '選擇米色', '大地色系'],
                'navy': ['海軍藍', '深藍色', 'navy'],
                'burgundy': ['酒紅色', '深紅色', '勃根地紅'],
                'cream': ['奶油色', '米白色', 'cream']
            }

            for color, phrases in ai_color_suggestions.items():
                for phrase in phrases:
                    if phrase in bot_response:
                        ai_recommendations['suggested_colors'].append(color)
                        break

            # 提取AI建議的風格
            style_suggestions = {
                'layered': ['層次', '疊穿', '多層次', 'layer'],
                'minimalist': ['簡約', '簡潔', '乾淨', 'minimal'],
                'professional': ['專業', '正式', '商務', 'professional'],
                'casual_chic': ['休閒優雅', 'casual chic', '輕鬆時尚'],
                'polished': ['精緻', '優雅', '有質感', 'polished'],
                'structured': ['有結構', '俐落', 'structured'],
                'flowing': ['飄逸', '柔軟', 'flowing'],
                'tailored': ['合身', '剪裁', 'tailored']
            }

            for style_type, keywords in style_suggestions.items():
                for keyword in keywords:
                    if keyword in bot_response:
                        ai_recommendations['suggested_styles'].append(style_type)
                        break

            # 提取搭配技巧
            styling_tips = [
                '腰帶', '配件', '項鍊', '耳環', '手錶', '包包', '鞋子',
                '捲袖', '紮衣角', '開襟', '半塞', '全塞'
            ]

            for tip in styling_tips:
                if tip in bot_response:
                    ai_recommendations['styling_tips'].append(tip)

    # 去除重複項目
    for key in ai_recommendations:
        ai_recommendations[key] = list(set(ai_recommendations[key]))

    print(f"提取的AI建議：{ai_recommendations}")
    return ai_recommendations

# 分析使用者偏好
def analyze_user_preferences(history):

    user_prefs = {
        'preferred_colors': [],
        'preferred_styles': [],
        'occasions': [],
        'fit_preferences': [],
        'personal_traits': []
    }

    if not history:
        return user_prefs

    # 分析使用者的表達
    full_conversation = ""
    for entry in history:
        if isinstance(entry, list) and len(entry) >= 1:
            user_msg = entry[0] if entry[0] else ""
            full_conversation += f" {user_msg}"

    full_conversation = full_conversation.lower()

    # 使用者的顏色偏好
    user_colors = {
        'black': ['我喜歡黑色', '黑色', '全黑', '偏愛黑'],
        'white': ['我喜歡白色', '白色', '純白'],
        'blue': ['我喜歡藍色', '藍色', '深藍'],
        'pink': ['我喜歡粉色', '粉色', '粉紅'],
        'red': ['我喜歡紅色', '紅色']
    }

    for color, expressions in user_colors.items():
        for expr in expressions:
            if expr in full_conversation:
                user_prefs['preferred_colors'].append(color)
                break

    # 使用者的風格偏好
    user_styles = {
        'sweet': ['甜美', '可愛', '少女'],
        'cool': ['帥氣', '酷', '個性'],
        'elegant': ['優雅', '氣質', '高雅'],
        'casual': ['休閒', '輕鬆', '舒適'],
        'minimalist': ['簡約', '簡單', '不複雜']
    }

    for style, keywords in user_styles.items():
        for keyword in keywords:
            if keyword in full_conversation:
                user_prefs['preferred_styles'].append(style)
                break

    print(f"👤 用戶偏好：{user_prefs}")
    return user_prefs

# 整合使用者偏好，生成prompt
def create_comprehensive_fashion_prompt(weather, user_prefs, ai_recommendations, gender="neutral"):

    temp = weather['temperature']
    gender_map = {"女性": "woman", "男性": "man", "中性": "person"}
    gender_en = gender_map.get(gender, "person")

    # 天氣資訊
    if temp >= 28:
        weather_base = "light summer clothing, breathable fabrics, short sleeves"
        weather_context = "hot summer day"
    elif temp >= 22:
        weather_base = "comfortable spring outfit, light layers"
        weather_context = "mild spring weather"
    elif temp >= 15:
        weather_base = "autumn clothing, long sleeves, light jacket"
        weather_context = "cool autumn day"
    else:
        weather_base = "warm winter outfit, cozy layers, warm jacket"
        weather_context = "cold winter day"

    # 服裝
    ai_clothing_elements = []
    if ai_recommendations['suggested_items']:
        items = ai_recommendations['suggested_items']
        clothing_descriptions = {
            'blazer': 'professional blazer',
            'shirt': 'elegant shirt',
            'sweater': 'cozy sweater',
            'dress': 'stylish dress',
            'pants': 'well-fitted pants',
            'jeans': 'classic jeans',
            'skirt': 'fashionable skirt',
            'jacket': 'stylish jacket',
            'cardigan': 'comfortable cardigan',
            'blouse': 'elegant blouse',
            'coat': 'sophisticated coat'
        }

        for item in items[:3]:  # 最多取3個主要單品
            if item in clothing_descriptions:
                ai_clothing_elements.append(clothing_descriptions[item])

    # 整合AI建議的顏色
    color_priority = []

    # 優先考慮AI建議的顏色
    if ai_recommendations['suggested_colors']:
        ai_colors = ai_recommendations['suggested_colors'][:2]  # 最多2個AI建議顏色
        color_priority.extend(ai_colors)

    # 其次考慮使用者偏好顏色
    if user_prefs['preferred_colors'] and len(color_priority) < 2:
        user_colors = [c for c in user_prefs['preferred_colors'] if c not in color_priority]
        color_priority.extend(user_colors[:2-len(color_priority)])

    # 整合AI建議的風格
    style_elements = []

    # AI專業風格建議
    if ai_recommendations['suggested_styles']:
        ai_style_map = {
            'layered': 'layered styling',
            'minimalist': 'clean minimalist aesthetic',
            'professional': 'polished professional look',
            'casual_chic': 'effortlessly chic style',
            'polished': 'refined elegant finish',
            'structured': 'well-structured silhouette',
            'flowing': 'graceful flowing lines',
            'tailored': 'perfectly tailored fit'
        }

        for ai_style in ai_recommendations['suggested_styles'][:2]:
            if ai_style in ai_style_map:
                style_elements.append(ai_style_map[ai_style])

    # 融入使用者風格偏好
    if user_prefs['preferred_styles']:
        user_style_map = {
            'sweet': 'sweet feminine touch',
            'cool': 'cool edgy vibe',
            'elegant': 'sophisticated elegance',
            'casual': 'relaxed comfortable style',
            'minimalist': 'minimalist simplicity'
        }

        for user_style in user_prefs['preferred_styles'][:1]:  # 用戶風格作為輔助
            if user_style in user_style_map and user_style_map[user_style] not in style_elements:
                style_elements.append(user_style_map[user_style])

    # 組合完整描述
    outfit_components = []

    # 天氣基礎
    outfit_components.append(weather_base)

    # AI建議的單品
    if ai_clothing_elements:
        outfit_components.append(", ".join(ai_clothing_elements))

    # 整合顏色方案
    if color_priority:
        if len(color_priority) == 1:
            color_desc = f"primarily {color_priority[0]} colored clothing"
        else:
            color_desc = f"{color_priority[0]} and {color_priority[1]} color palette"
        outfit_components.append(color_desc)

    # 風格描述
    if style_elements:
        outfit_components.append(", ".join(style_elements))

    # 搭配技巧
    if ai_recommendations['styling_tips']:
        tips = ai_recommendations['styling_tips'][:2]
        styling_desc = f"styled with attention to {', '.join(tips)}"
        outfit_components.append(styling_desc)

    # 最終組合
    complete_outfit_description = ", ".join(outfit_components)

    # 生成正面提示詞
    positive_prompt = f"""
    A single {gender_en} wearing {complete_outfit_description},
    perfect for {weather_context}, full body portrait,
    fashion photography, high quality, detailed,
    professional studio lighting, clean background,
    coordinated outfit following stylist recommendations,
    natural confident pose, perfect anatomy,
    one person only, complete outfit, fashion model,
    stylish, photorealistic, well-balanced composition
    """.strip().replace('\n', ' ').replace('  ', ' ')

    # 負面提示詞
    negative_prompt = """
    multiple people, extra limbs, extra arms, extra legs,
    extra hands, extra fingers, missing limbs, missing arms,
    deformed hands, deformed fingers, weird hands,
    too many fingers, too few fingers, poorly drawn hands,
    bad anatomy, ugly, distorted, blurry, low quality,
    bad proportions, unfashionable, messy, inappropriate clothing,
    bad fit, duplicate, malformed, mutation, cropped,
    worst quality, jpeg artifacts, watermark, clashing colors,
    mismatched outfit, unstylish combination
    """.strip().replace('\n', ' ').replace('  ', ' ')

    print(f"整合提示詞 - AI建議: {len(ai_recommendations['suggested_items'])}項單品, {len(ai_recommendations['suggested_colors'])}個顏色")
    print(f"用戶偏好 - {len(user_prefs['preferred_colors'])}個顏色偏好, {len(user_prefs['preferred_styles'])}種風格")
    print(f"最終描述：{complete_outfit_description[:100]}...")

    return positive_prompt, negative_prompt

def generate_outfit_image_final(weather, user_context, gender, history):

    if pipe is None:
        return None, "圖片生成功能目前無法使用，Stable Diffusion模型未載入成功"

    try:
        print("開始生成穿搭圖片...")

        # 分析AI顧問建議
        ai_recommendations = extract_ai_recommendations(history)

        # 分析使用者偏好
        user_preferences = analyze_user_preferences(history)

        # 生成整合提示詞
        positive_prompt, negative_prompt = create_comprehensive_fashion_prompt(
            weather, user_preferences, ai_recommendations, gender
        )

        print(f"正在生成整合AI建議和個人偏好的專屬穿搭圖...")

        # 記憶體管理
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        # 動態種子
        seed_string = f"{time.time()}_{positive_prompt[:50]}_{len(history)}"
        seed_hash = hashlib.md5(seed_string.encode()).hexdigest()
        dynamic_seed = int(seed_hash[:8], 16) % (2**32)

        generator = torch.Generator(device).manual_seed(dynamic_seed)

        with torch.no_grad():
            result = pipe(
                prompt=positive_prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=CONFIG['IMAGE_GENERATION']['INFERENCE_STEPS'],
                guidance_scale=CONFIG['IMAGE_GENERATION']['GUIDANCE_SCALE'],
                width=CONFIG['IMAGE_GENERATION']['WIDTH'],
                height=CONFIG['IMAGE_GENERATION']['HEIGHT'],
                generator=generator,
                num_images_per_prompt=1
            )

        # 清理記憶體
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        return result.images[0], f"根據AI專業建議和您的個人喜好生成的穿搭圖！(種子: {dynamic_seed})"

    except torch.cuda.OutOfMemoryError:
        # GPU記憶體不足處理
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
        return None, "GPU記憶體不足，請稍後再試或重新啟動 Runtime"
    except Exception as e:
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
        print(f"圖片生成錯誤詳情: {e}")
        return None, f"圖片生成失敗：{str(e)}"

###11. 主要應用邏輯
把前面所有功能整合在一起，建立主要的聊天處理函數和圖片生成函數。這些函數會被Gradio界面呼叫。

In [ ]:
def fashion_advisor_chat(message, gender, city, history):
    global conversation_manager

    if not message.strip():
        # 重置對話管理器
        conversation_manager = ConversationStateManager()

        # 獲取天氣資訊來個性化開場
        weather = get_weather_info(city)
        temp = weather['temperature']

        welcome_msg = f"哈囉！我是你的穿搭顧問 ✨\n\n今天{temp}度，要去哪裡呢？"
        return history + [("", welcome_msg)], history + [("", welcome_msg)]

    try:
        print(f"收到用戶訊息: {message}")

        # 獲取天氣
        weather = get_weather_info(city)

        response = generate_smart_fashion_response(message, weather, history, conversation_manager)

        new_history = history + [(message, response)]
        print(f"回應生成成功，對話長度: {len(new_history)}")
        return new_history, new_history

    except Exception as e:
        print(f"對話處理錯誤: {e}")
        error_msg = f"不好意思，系統出了點問題。讓我們重新開始吧！"
        new_history = history + [(message, error_msg)]
        return new_history, new_history

def generate_fashion_image_from_chat(chat_history, gender, city):

    if not chat_history:
        return None, "請先跟我聊聊你的需求喔！", ""

    try:
        print("開始分析對話記錄...")

        # 提取最新對話
        latest_user_message = ""
        for user_msg, _ in reversed(chat_history):
            if user_msg and user_msg.strip():
                latest_user_message = user_msg
                break

        if not latest_user_message:
            return None, "找不到對話內容", ""

        weather = get_weather_info(city)
        user_context = analyze_conversation_context(chat_history, latest_user_message)

        # 分析資訊
        analysis_info = f"""
**分析你的需求：**

**你說：** "{latest_user_message[:80]}{'...' if len(latest_user_message) > 80 else ''}"

**我了解到：**
"""

        if user_context.get('has_occasion'):
            analysis_info += f"• 場合：{user_context['occasion']}\n"
        if user_context.get('has_mood'):
            analysis_info += f"• 心情：{user_context['mood']}\n"
        if user_context.get('has_style'):
            analysis_info += f"• 風格：{user_context['style']}\n"

        analysis_info += f"• 天氣：{weather['temperature']}°C，{weather['description']}\n"

        # 生成圖片
        print("開始生成穿搭圖片...")
        image, status_msg = generate_outfit_image_final(weather, user_context, gender, chat_history)

        # 分析AI顧問建議和使用者偏好
        ai_recommendations = extract_ai_recommendations(chat_history)
        user_preferences = analyze_user_preferences(chat_history)

        # 生成prompt資訊
        positive_prompt, negative_prompt = create_comprehensive_fashion_prompt(
            weather, user_preferences, ai_recommendations, gender
        )

        prompt_info = f"""
**圖片生成prompt：**

**positive prompt：**
{positive_prompt}

**negative prompt：**
{negative_prompt}
        """

        return image, f"{analysis_info}\n\n{status_msg}", prompt_info

    except Exception as e:
        print(f"圖片生成流程錯誤: {e}")
        return None, f"生成圖片時出錯：{str(e)}", ""

### 12. 打造 Gradio Web App
用Gradio建立網頁介面，讓使用者可以透過瀏覽器使用這個穿搭顧問。介面左邊是聊天區，右邊是圖片顯示區。

In [ ]:
custom_css = """
.container {
    max-width: 1400px;
    margin: auto;
}
.header {
    text-align: center;
    padding: 30px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    border-radius: 15px;
    margin-bottom: 30px;
    box-shadow: 0 8px 32px rgba(0,0,0,0.1);
}
.chat-container {
    height: 400px;
    overflow-y: auto;
    border-radius: 10px;
}
.settings-panel {
    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
    padding: 20px;
    border-radius: 15px;
    margin-bottom: 20px;
    border: 1px solid #e5e7eb;
}
.example-buttons {
    background-color: #f8fafc;
    padding: 15px;
    border-radius: 10px;
    margin-top: 20px;
}
.status-box {
    background-color: #f0f9ff;
    border-left: 4px solid #0ea5e9;
    padding: 15px;
    border-radius: 8px;
}
"""

# 主要 Gradio 介面
with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="AI穿搭顧問") as app:

    with gr.Column(elem_classes="container"):
        gr.HTML("""
        <div class="header">
            <h1>👗 AI穿搭顧問 🎨</h1>
        </div>
        """)

        # 設定區域
        with gr.Row(elem_classes="settings-panel"):
            with gr.Column(scale=1):
                gender_choice = gr.Radio(
                    choices=["女性", "男性", "中性"],
                    value="中性",
                    label="👤 性別設定"
                )
            with gr.Column(scale=1):
                city_input = gr.Textbox(
                    value="Taipei",
                    label="🌍 所在城市",
                    placeholder="輸入城市名稱（英文，如：Tokyo, London）"
                )

        with gr.Row():
            # 對話區域
            with gr.Column(scale=2):
                gr.Markdown("## 💭 聊聊你的穿搭需求")

                chatbot = gr.Chatbot(
                    height=400,
                    elem_classes="chat-container",
                    avatar_images=["👤", "👗"]
                )

                with gr.Row():
                    msg_input = gr.Textbox(
                        placeholder="今天要做什麼呢？想穿什麼風格？",
                        label="",
                        lines=2,
                        scale=4
                    )
                    send_btn = gr.Button("送出", scale=1, variant="primary")

                with gr.Row():
                    generate_btn = gr.Button(
                        "生成我的穿搭圖",
                        variant="primary",
                        size="lg",
                        scale=2
                    )
                    clear_btn = gr.Button("重新開始", variant="secondary", scale=1)

            # 圖片區域
            with gr.Column(scale=2):
                gr.Markdown("## 🎨 你的專屬穿搭圖")

                fashion_image = gr.Image(
                    height=450,
                    elem_classes="fashion-image"
                )

    # 對話狀態
    chat_history = gr.State([])

    # 事件綁定函數
    def send_message(message, gender, city, history):
        if not message.strip():
            return history, history, ""

        result_history, state_history = fashion_advisor_chat(message, gender, city, history)

        # 顯示當前對話狀態
        global conversation_manager
        summary = conversation_manager.get_conversation_summary()

        print(f"📊 對話狀態更新:")
        print(f"   輪次: {summary['turns']}")
        print(f"   準備度: {summary['readiness_score']}%")
        print(f"   收集資訊: {conversation_manager.collected_info}")

        return result_history, state_history, ""

    def clear_conversation():
        """清除對話記錄"""
        global conversation_manager
        conversation_manager = ConversationStateManager()
        print("🔄 對話已重置")
        return [], [], None

    # 事件綁定
    send_btn.click(
        fn=send_message,
        inputs=[msg_input, gender_choice, city_input, chat_history],
        outputs=[chatbot, chat_history, msg_input]
    )

    msg_input.submit(
        fn=send_message,
        inputs=[msg_input, gender_choice, city_input, chat_history],
        outputs=[chatbot, chat_history, msg_input]
    )

    clear_btn.click(
        fn=clear_conversation,
        outputs=[chatbot, chat_history, fashion_image]
    )

    generate_btn.click(
        fn=lambda history, gender, city: generate_fashion_image_from_chat(history, gender, city)[0],  # 只返回圖片
        inputs=[chat_history, gender_choice, city_input],
        outputs=[fashion_image]
    )

if __name__ == "__main__":
    app.launch(
        share=True,
        debug=True,
        server_name="0.0.0.0",
        server_port=7860,
        show_error=True
    )